In [1]:
!pip install gymnasium stable-baselines3[extra] --quiet
import os
from google.colab import files

print("[INFO] Please upload your kaggle.json file...")
uploaded = files.upload()

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
!mv kaggle.json {kaggle_dir}/kaggle.json
!chmod 600 {kaggle_dir}/kaggle.json

DATASET_SLUG = 'dhoogla/cicids2017'
DOWNLOAD_DIR = '/content/cicids2017'
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

print(f"[INFO] Downloading {DATASET_SLUG}...")
!kaggle datasets download -d {DATASET_SLUG} -p {DOWNLOAD_DIR}
!unzip -q {DOWNLOAD_DIR}/cicids2017.zip -d {DOWNLOAD_DIR}
print("[INFO] Download complete. Ready for orchestration.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 19.6 MB/s eta 0:00:00
[INFO] Please upload your kaggle.json file...


Saving kaggle.json to kaggle.json
[INFO] Downloading dhoogla/cicids2017...
Dataset URL: https://www.kaggle.com/datasets/dhoogla/cicids2017
License(s): CC-BY-NC-SA-4.0
100% 227M/227M [00:01<00:00, 159MB/s]

[INFO] Download complete. Ready for orchestration.


In [2]:
%%writefile train_logger.py
from stable_baselines3.common.callbacks import BaseCallback

class TrainLoggerCallback(BaseCallback):
    def __init__(self, verbose=0):
        super(TrainLoggerCallback, self).__init__(verbose)
        self.episode_count = 0

    def _on_step(self) -> bool:
        # Check if an episode finished
        done = self.locals.get("dones")[0]
        if done:
            self.episode_count += 1
            # Extract reward from info if available or environment
            ep_info = self.locals.get("infos")[0].get("episode")
            if ep_info:
                print(f"[EPISODE {self.episode_count}] Reward: {ep_info['r']:.2f} | Length: {ep_info['l']}")
            else:
                print(f"[EPISODE {self.episode_count}] Episode finished at step {self.num_timesteps}")
        return True

Writing train_logger.py


In [3]:
import sys

# Read the existing train.py
with open('train.py', 'r') as f:
    lines = f.readlines()

# Modify it to include the callback
new_content = []
inserted_import = False
for line in lines:
    if "from dqn_agent import" in line and not inserted_import:
        new_content.append(line)
        new_content.append("from train_logger import TrainLoggerCallback\n")
        inserted_import = True
    elif "dqn_agent.learn(" in line:
        # Replace the learn call to include the callback
        new_line = line.replace("progress_bar=True)", "progress_bar=True, callback=TrainLoggerCallback())")
        new_content.append(new_line)
    else:
        new_content.append(line)

with open('train.py', 'w') as f:
    f.writelines(new_content)

print("[INFO] train.py updated with Episode Logging Callback.")

[INFO] train.py updated with Episode Logging Callback.


In [4]:
!python train.py

2026-05-20 19:16:47.759536: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
--- Phase 1: Data Preprocessing ---
[INFO] Loading raw data files...
[INFO] Sampling 100000 rows...
[INFO] Cleaning data and dropping metadata...
[INFO] Scaling features...
[INFO] Scaler saved to ids_minmax_scaler.pkl

--- Phase 2: Train/Test Split ---
[INFO] Exporting test_traffic.csv for live FastAPI dashboard...


In [5]:
import zipfile
from google.colab import files

zip_name = 'fastapi_artifacts.zip'
files_to_zip = [
    'ids_minmax_scaler.pkl',
    'ids_rf_baseline.pkl',
    'dqn_ids_model.zip',
    'test_traffic.csv',
    'rf_evaluation.png',
    'dqn_evaluation.png'
]

with zipfile.ZipFile(zip_name, 'w') as zf:
    for f in files_to_zip:
        if os.path.exists(f):
            zf.write(f)

print(f"[INFO] Downloading {zip_name} to your local machine...")
files.download(zip_name)

[INFO] Downloading fastapi_artifacts.zip to your local machine...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>